### **Final System Demo**

This demo shows the complete end-to-end big data system for heart disease prediction.

It demonstrates:
- data loading and preprocessing
- machine learning prediction model
- streaming simulation (kafka-style data flow)
- adversarial data impact on model performance

The goal is to simulate how a real big data system behaves from ingestion to prediction under normal and noisy conditions.

In [15]:
print("=== BIG DATA SYSTEM DEMO STARTED ===")
print("Pipeline: Data → Kafka → Stream → ML Prediction")

=== BIG DATA SYSTEM DEMO STARTED ===
Pipeline: Data → Kafka → Stream → ML Prediction


### Step 1: Import Libraries

In [16]:
# import required libraries for data processing, modeling, and simulation

import pandas as pd
import numpy as np
import time
from collections import deque

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

import warnings
warnings.filterwarnings("ignore")

print("libraries loaded successfully")

libraries loaded successfully


### Step 2: Load Dataset

In [17]:
df = pd.read_csv("/content/cleaned.csv")

print("Dataset loaded ✔")
print("Shape:", df.shape)

df.head()

Dataset loaded ✔
Shape: (297, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,1
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


### Step 3: prepare machine learning model

In [18]:
# separate features and target variable

x = df.drop("target", axis=1)
y = df["target"]

# scale features for better model performance
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)

# train logistic regression model
model = LogisticRegression()
model.fit(x_scaled, y)

print("model trained successfully")

model trained successfully


### Step 4: baseline model evaluation

In [19]:
# evaluate model on clean dataset

predictions = model.predict(x_scaled)

accuracy = accuracy_score(y, predictions)

print("baseline model accuracy:", round(accuracy, 4))

baseline model accuracy: 0.8485


### Step 5: simulate streaming data (kafka style)

In [21]:
# simulate real-time incoming data stream
# this represents kafka-like message ingestion

stream_buffer = deque(maxlen=5)

print("starting data stream simulation...\n")

for i in range(10):
    # simulate receiving one patient record at a time
    record = df.iloc[i % len(df)].to_dict()
    stream_buffer.append(record)

    print("new incoming record:", record)

    time.sleep(0.5)

starting data stream simulation...

new incoming record: {'age': 63.0, 'sex': 1.0, 'cp': 1.0, 'trestbps': 145.0, 'chol': 233.0, 'fbs': 1.0, 'restecg': 2.0, 'thalach': 150.0, 'exang': 0.0, 'oldpeak': 2.3, 'slope': 3.0, 'ca': 0.0, 'thal': 6.0, 'target': 0.0}
new incoming record: {'age': 67.0, 'sex': 1.0, 'cp': 4.0, 'trestbps': 160.0, 'chol': 286.0, 'fbs': 0.0, 'restecg': 2.0, 'thalach': 108.0, 'exang': 1.0, 'oldpeak': 1.5, 'slope': 2.0, 'ca': 3.0, 'thal': 3.0, 'target': 1.0}
new incoming record: {'age': 67.0, 'sex': 1.0, 'cp': 4.0, 'trestbps': 120.0, 'chol': 229.0, 'fbs': 0.0, 'restecg': 2.0, 'thalach': 129.0, 'exang': 1.0, 'oldpeak': 2.6, 'slope': 2.0, 'ca': 2.0, 'thal': 7.0, 'target': 1.0}
new incoming record: {'age': 37.0, 'sex': 1.0, 'cp': 3.0, 'trestbps': 130.0, 'chol': 250.0, 'fbs': 0.0, 'restecg': 0.0, 'thalach': 187.0, 'exang': 0.0, 'oldpeak': 3.5, 'slope': 3.0, 'ca': 0.0, 'thal': 3.0, 'target': 0.0}
new incoming record: {'age': 41.0, 'sex': 0.0, 'cp': 2.0, 'trestbps': 130.0, 'ch

### Step 6: stream processing and prediction

In [22]:
# process streamed data and make predictions in real time

print("\nprocessing streamed data...\n")

for i, record in enumerate(stream_buffer):

    # convert record to model input format
    x_input = np.array(list(record.values())[:-1]).reshape(1, -1)
    x_input_scaled = scaler.transform(x_input)

    # make prediction
    prediction = model.predict(x_input_scaled)[0]

    if prediction == 1:
        result = "high risk"
    else:
        result = "low risk"

    print("patient", i + 1, "prediction:", result)


processing streamed data...

patient 1 prediction: low risk
patient 2 prediction: high risk
patient 3 prediction: low risk
patient 4 prediction: high risk
patient 5 prediction: high risk


### Step 7: adversarial data simulation

In [27]:
df_adv = df.copy()

# feature corruption (sensor noise simulation)
noise = np.random.normal(0, 30, df_adv.shape[0])
df_adv["trestbps"] += noise
df_adv["chol"] += noise * 0.5

# label poisoning (data attack simulation)
flip_mask = np.random.rand(len(df_adv)) < 0.25
df_adv.loc[flip_mask, "target"] = 1 - df_adv.loc[flip_mask, "target"]

# prepare adversarial dataset
x_adv = df_adv.drop("target", axis=1)
x_adv_scaled = scaler.transform(x_adv)

# predictions
adv_predictions = model.predict(x_adv_scaled)

# accuracy
adv_accuracy = accuracy_score(y, adv_predictions)

print("adversarial model accuracy:", round(adv_accuracy, 4))
print("accuracy drop:", round(accuracy - adv_accuracy, 4))

adversarial model accuracy: 0.835
accuracy drop: 0.0135


After introducing feature corruption and label poisoning, the model accuracy decreased, showing that the system is sensitive to adversarial data conditions, which reflects real-world risks in healthcare systems.

### Step 8: system summary

In [24]:
# final summary of the entire system

print("\nsystem summary")
print("----------------------------")

print("data pipeline: working")
print("machine learning model: working")
print("stream simulation: working")
print("adversarial testing: working")

print("\nresults:")
print("baseline accuracy:", round(accuracy, 4))
print("adversarial accuracy:", round(adv_accuracy, 4))

print("\nend-to-end system demo completed successfully")


system summary
----------------------------
data pipeline: working
machine learning model: working
stream simulation: working
adversarial testing: working

results:
baseline accuracy: 0.8485
adversarial accuracy: 0.8519

end-to-end system demo completed successfully


This concludes the final system demonstration.

The notebook shows a complete big data pipeline including:
- data ingestion
- machine learning prediction
- streaming simulation
- adversarial robustness testing

This demonstrates how a real-world healthcare analytics system behaves under normal and degraded data conditions.